# Titanic Survival Prediction

This notebook builds a Random Forest model to predict Titanic survival. It emphasizes feature creation (Title, Family Size, Cabin Presence), missing value imputation, and model explainability using SHAP and Feature Importance.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

## 1. Data Preprocessing and Feature Engineering

We extract Title from Name, calculate Family Size, and create a Has_Cabin indicator.

In [ ]:
def extract_title(name):
    title_search = re.search(r' ([A-Za-z]+)\.', name)
    if title_search:
        return title_search.group(1)
    return ""

def preprocess_data(df):
    df = df.copy()
    
    # 1. Extract Title
    df['Title'] = df['Name'].apply(extract_title)
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # 2. Family Size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    
    # 3. Cabin Presence
    df['Has_Cabin'] = df['Cabin'].apply(lambda x: 0 if pd.isna(x) else 1)
    
    # Fill missing Age by Title median
    title_age_median = df.groupby('Title')['Age'].median()
    df['Age'] = df.apply(lambda row: title_age_median[row['Title']] if pd.isna(row['Age']) else row['Age'], axis=1)
    
    # Impute missing Fare with median
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # Impute missing Embarked with mode
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # Drop unnecessary columns
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df = df.drop(columns=drop_cols)
    
    return df

## 2. Model Pipeline Setup

In [ ]:
def build_pipeline():
    categorical_features = ['Sex', 'Embarked', 'Title']
    numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'Has_Cabin']

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)])

    clf = Pipeline(steps=[('preprocessor', preprocessor),
                          ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))])
    
    return clf, numeric_features, categorical_features

## 3. Training and Evaluation

In [ ]:
# Load dataset
df = pd.read_csv('titanic.csv')
df_processed = preprocess_data(df)

X = df_processed.drop('Survived', axis=1)
y = df_processed['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline, num_features, cat_features = build_pipeline()
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 4. Explainability (Feature Importance & SHAP)

In [ ]:
clf = pipeline.named_steps['classifier']
preprocessor = pipeline.named_steps['preprocessor']

cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(cat_features)
feature_names = num_features + list(cat_feature_names)

importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(len(importances)), importances[indices], align="center")
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.xlim([-1, len(importances)])
plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    X_train_transformed = preprocessor.transform(X_train)
    if hasattr(X_train_transformed, 'toarray'):
        X_train_transformed = X_train_transformed.toarray()
    
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_train_transformed)
    
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
        
    shap.summary_plot(shap_values, X_train_transformed, feature_names=feature_names)
except ImportError:
    print("SHAP is not installed.")

## 5. Save Model

In [ ]:
joblib.dump(pipeline, 'titanic_model.pkl')
print("Model saved to 'titanic_model.pkl'")